In [60]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
import pickle

In [61]:
df=pd.read_csv('..//data//processed//merged_dataset.csv')
df.head()

,sample_id,label,timestep,flex1,flex2,flex3,flex4,flex5,accel_x,accel_y,accel_z,gyro_x,gyro_y,gyro_z,person
0,1,salam,0,4046,3409,3721,3479,3581,4296,367,-442,210,-27,-32,Apurbo
1,1,salam,1,4095,3417,3733,3498,3611,3903,711,-229,-1360,148,-286,Apurbo
2,1,salam,2,4095,3414,3749,3502,3605,4200,38,-1073,2438,-488,173,Apurbo
3,1,salam,3,4095,3411,3725,3487,3609,4059,800,-560,1720,-611,-317,Apurbo
4,1,salam,4,4095,3413,3734,3499,3614,4583,1136,-1724,1967,-882,-1160,Apurbo


In [62]:
# ============================================
# 1. UNDERSTAND YOUR DATA
# ============================================

print("Dataset Info:")
print(f"Total entries: {len(df)}")
print(f"Unique samples: {df['sample_id'].nunique()}")
print(f"Unique labels: {df['label'].nunique()}")
print(f"Samples per label:\n{df.groupby('label')['sample_id'].nunique()}")

Dataset Info:
Total entries: 4400
Unique samples: 220
Unique labels: 11
Samples per label:
label
achen        20
achi         20
ami          20
apnar        20
apni         20
dhonnobad    20
kemon        20
kii          20
naam         20
salam        20
valo         20
Name: sample_id, dtype: int64


In [63]:
# ============================================
# 2. CONVERT TO SAMPLES (20 timesteps × 11 features)
# ============================================

# Define feature columns
feature_cols = ['flex1', 'flex2', 'flex3', 'flex4', 'flex5', 
                'accel_x', 'accel_y', 'accel_z', 
                'gyro_x', 'gyro_y', 'gyro_z']

# Create samples
X_samples = []
y_samples = []

for sample_id in df['sample_id'].unique():
    # Get all data for this sample
    sample_data = df[df['sample_id'] == sample_id]
    
    # Sort by timestep to maintain order (0-19)
    sample_data = sample_data.sort_values('timestep')
    
    # Extract features (20 timesteps × 11 features)
    features = sample_data[feature_cols].values  # Shape: (20, 11)
    
    # Get label
    label = sample_data['label'].iloc[0]
    
    X_samples.append(features)
    y_samples.append(label)

# Convert to numpy arrays
X_samples = np.array(X_samples)  # Shape: (220, 20, 11)
y_samples = np.array(y_samples)  # Shape: (220,)

print(f"\nX_samples shape: {X_samples.shape}")  # (220, 20, 11)
print(f"y_samples shape: {y_samples.shape}")    # (220,)
print(f"Number of classes: {len(np.unique(y_samples))}")


X_samples shape: (220, 20, 11)
y_samples shape: (220,)
Number of classes: 11


In [64]:
# ============================================
# 3. SPLIT DATA (80% train, 20% test)
# ============================================

X_train, X_test, y_train, y_test = train_test_split(
    X_samples, y_samples,
    test_size=0.2,
    random_state=42,
    stratify=y_samples
)

print(f"\nX_train shape: {X_train.shape}")  # (176, 20, 11)
print(f"X_test shape: {X_test.shape}")      # (44, 20, 11)
print(f"y_train shape: {y_train.shape}")    # (176,)
print(f"y_test shape: {y_test.shape}")      # (44,)

# Check class distribution
print(f"\nTraining samples per class:\n{pd.Series(y_train).value_counts().sort_index()}")
print(f"\nTest samples per class:\n{pd.Series(y_test).value_counts().sort_index()}")


X_train shape: (176, 20, 11)
X_test shape: (44, 20, 11)
y_train shape: (176,)
y_test shape: (44,)

Training samples per class:
achen        16
achi         16
ami          16
apnar        16
apni         16
dhonnobad    16
kemon        16
kii          16
naam         16
salam        16
valo         16
Name: count, dtype: int64

Test samples per class:
achen        4
achi         4
ami          4
apnar        4
apni         4
dhonnobad    4
kemon        4
kii          4
naam         4
salam        4
valo         4
Name: count, dtype: int64


In [65]:

# ============================================
# 4. LABEL ENCODING
# ============================================

label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)

print(f"\nClasses: {label_encoder.classes_}")
print(f"y_train_encoded shape: {y_train_encoded.shape}")
print(f"y_test_encoded shape: {y_test_encoded.shape}")



Classes: ['achen' 'achi' 'ami' 'apnar' 'apni' 'dhonnobad' 'kemon' 'kii' 'naam'
 'salam' 'valo']
y_train_encoded shape: (176,)
y_test_encoded shape: (44,)


In [66]:
#save the label encoder for later use in the Models directory as label_encoder.pkl
with open('../Models/label_encoder.pkl', 'wb') as f:
    pickle.dump(label_encoder, f)
    

In [67]:
# ============================================
# 5. SCALING - RESHAPE, SCALE, RESHAPE BACK
# ============================================

scaler = StandardScaler()

# Get shapes
n_train_samples, n_timesteps, n_features = X_train.shape
n_test_samples, _, _ = X_test.shape

print(f"\nScaling shapes:")
print(f"  Train: {n_train_samples} samples × {n_timesteps} timesteps × {n_features} features")
print(f"  Test: {n_test_samples} samples × {n_timesteps} timesteps × {n_features} features")

# Reshape to 2D for scaling: (samples * timesteps, features)
X_train_2d = X_train.reshape(-1, n_features)  # Shape: (3520, 11)
X_test_2d = X_test.reshape(-1, n_features)    # Shape: (880, 11)

print(f"  Train 2D shape: {X_train_2d.shape}")
print(f"  Test 2D shape: {X_test_2d.shape}")

# Fit scaler on training data
X_train_scaled_2d = scaler.fit_transform(X_train_2d)
X_test_scaled_2d = scaler.transform(X_test_2d)

# Reshape back to 3D
X_train_scaled = X_train_scaled_2d.reshape(n_train_samples, n_timesteps, n_features)
X_test_scaled = X_test_scaled_2d.reshape(n_test_samples, n_timesteps, n_features)

print(f"\nScaled X_train shape: {X_train_scaled.shape}")
print(f"Scaled X_test shape: {X_test_scaled.shape}")


Scaling shapes:
  Train: 176 samples × 20 timesteps × 11 features
  Test: 44 samples × 20 timesteps × 11 features
  Train 2D shape: (3520, 11)
  Test 2D shape: (880, 11)

Scaled X_train shape: (176, 20, 11)
Scaled X_test shape: (44, 20, 11)


In [68]:


with open('../data/processed/preprocessed_data.pkl', 'wb') as f:
    pickle.dump({
        'X_train': X_train_scaled,
        'X_test': X_test_scaled,
        'y_train': y_train_encoded,
        'y_test': y_test_encoded,
    }, f)